# A4 — Transfer Learning & Fine-Tuning (CPU-Friendly)
## Data Set: Oxford-IIIT Pet
- #### Feature Extraction vs Fine-Tuning | Performance Comparison
### Models: VGG16 • ResNet50 • DenseNet121
#### Solution Notebook

**Hardware assumption:** CPU-only laptops/PC are acceptable (training time may vary).  
**Dataset:** Oxford-IIIT Pet (37 classes)  
**Recommended settings:** `IMG_SIZE=(128,128)`, `BATCH_SIZE=32`, `EPOCHS=5-6` (CPU-friendly)

---

This solution notebook provides a clean reference implementation for:
- Building a reproducible `tf.data` pipeline
- Training **frozen-backbone** models (feature extraction)
- Running **one controlled fine-tuning** experiment
- Comparing results (accuracy, training time, parameter counts)


## Q0 — Setup (Ungraded)
#### Import libraries, set seeds, and verify TensorFlow / TFDS.

In [13]:
import os, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers
from tensorflow.keras.applications import resnet, vgg16, densenet

print("TensorFlow:", tf.__version__)
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

TensorFlow: 2.19.0


## ✅ Student Instructions (Start Here)

Your work begins in the **next code cells (Q1–Q9)** and continues by answering questions in the **Markdown cells (Q10–Q13)**. These correspond to the questions listed in the assignment description on Canvas. Complete each cell by following the instructions provided in the **preceding Markdown cells**.

Tasks:
- This assignment focuses on **transfer learning** with pretrained CNN backbones.
- You will train **three frozen-backbone models** for a fair comparison:
  - **VGG16** (frozen base)
  - **ResNet50** (frozen base)
  - **DenseNet121** (frozen base)
- Then run **one controlled fine-tuning experiment** (unfreeze a small portion of one backbone with a smaller learning rate).
- Keep your training CPU-feasible (use the recommended settings unless you have a GPU).

## Q1 — Load Dataset & Inspect
Use TFDS: `oxford_iiit_pet` and inspect shapes/classes.
### Student Tasks

- Load the Oxford-IIIT Pet dataset and split into ds_train (80%), ds_val (20%), and ds_test.

- Extract number of classes and class names from ds_info.

- Display one sample: image shape, label index, and class name.

In [14]:
#============================================================
#Question Q1 — Load Oxford-IIIT Pet Dataset (Fill in the Blanks)
#============================================================
#Complete the TODO sections to:
#1) Load the Oxford-IIIT Pet dataset using TensorFlow Datasets
#2) Create train/validation/test splits
#3) Extract class names and number of classes
#4) Print dataset information
#5) Inspect one example image and label
#============================================================

# TODO 1: Load dataset with train/validation/test splits
(ds_train, ds_val, ds_test), ds_info = tfds.load(
    "oxford_iiit_pet",
    split=["train[:80%]", "train[80%:]", "test"],
    as_supervised=True,
    with_info=True,
)

# TODO 2: Get number of classes
num_classes = ds_info.features["label"].num_classes

# TODO 3: Get class names
class_names = ds_info.features["label"].names

# TODO 4: Print dataset information
print("Num classes:", num_classes)
print("Example classes:", class_names[:5])

# TODO 5: Inspect one example from the training set
for x, y in ds_train.take(1):
    print("Raw image shape:", x.shape,
          "| label:", int(y),
          "| class:", class_names[int(y)])

Num classes: 37
Example classes: ['Abyssinian', 'american_bulldog', 'american_pit_bull_terrier', 'basset_hound', 'beagle']
Raw image shape: (500, 500, 3) | label: 33 | class: Sphynx


## Q2 — Preprocessing & `tf.data` Pipeline
Resize to **128×128**, normalize, add light augmentation for training.

### Student Tasks

- Set image size and batch size.

- Preprocess images by resizing and normalizing.

- Apply data augmentation to the training dataset.

- Create batched and prefetched train, validation, and test pipelines.

In [15]:
#============================================================
#Question Q2 — Image Preprocessing & Data Pipeline (Fill in the Blanks)
#============================================================
#Complete the TODO sections to:
#1) Define image size (128×128) and batch size (32 to 64)
#2) Implement preprocessing (resize + normalization)
#3) Implement simple data augmentation
#4) Build TensorFlow data pipelines
#5) Shuffle, batch, and prefetch the datasets
#============================================================

# TODO 1: Define image size and batch size
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# ------------------------------------------------------------
# TODO 2: Preprocessing function
# ------------------------------------------------------------
@tf.function
def preprocess(image, label):

    # Resize image
    image = tf.image.resize(image, IMG_SIZE)

    # Convert to float32
    image = tf.cast(image, tf.float32)

    # Normalize pixels
    image = image / 255.0

    return image, label

# ------------------------------------------------------------
# TODO 3: Data augmentation function
# ------------------------------------------------------------
@tf.function
def augment(image, label):

    # Random horizontal flip
    image = tf.image.random_flip_left_right(image)

    # Random brightness
    image = tf.image.random_brightness(image, max_delta=0.2)

    return image, label

# ------------------------------------------------------------
# TODO 4: Apply preprocessing and augmentation
# ------------------------------------------------------------
train_ds = ds_train.map(preprocess, num_parallel_calls=AUTOTUNE)\
                   .map(augment, num_parallel_calls=AUTOTUNE)

val_ds   = ds_val.map(preprocess, num_parallel_calls=AUTOTUNE)

test_ds  = ds_test.map(preprocess, num_parallel_calls=AUTOTUNE)

# ------------------------------------------------------------
# TODO 5: Shuffle, batch, and prefetch
# ------------------------------------------------------------
train_ds = train_ds.shuffle(1000, seed=SEED)\
                   .batch(BATCH_SIZE)\
                   .prefetch(AUTOTUNE)

val_ds   = val_ds.batch(BATCH_SIZE)\
                 .prefetch(AUTOTUNE)

test_ds  = test_ds.batch(BATCH_SIZE)\
                  .prefetch(AUTOTUNE)

print("Ready:", train_ds, val_ds, test_ds)

Ready: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))> <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))> <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>


## Q3 — Utilities: Compile / Train / Evaluate / Count Params

### Student Tasks

- Compile the model using the Adam optimizer, sparse categorical cross-entropy loss, and accuracy metric.

- Train the model for several epochs and measure the training time.

- Evaluate the trained model on the validation and test datasets and report accuracy.

In [16]:
# ============================================================
# Question Q3 — Model Training Utilities
# ============================================================

import numpy as np
import time
import tensorflow as tf

# ------------------------------------------------------------
# TODO 1: Compile the model
# ------------------------------------------------------------
def compile_model(model, lr=1e-3):

    model.compile(

        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),

        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),

        metrics=["accuracy"],
    )

    return model


# ------------------------------------------------------------
# TODO 2: Compute total parameters of the model
# ------------------------------------------------------------
def total_params(model):

    return int(np.sum([np.prod(v.shape) for v in model.trainable_variables])) + \
           int(np.sum([np.prod(v.shape) for v in model.non_trainable_variables]))


# ------------------------------------------------------------
# TODO 3: Train the model and measure training time
# ------------------------------------------------------------
def train_and_time(model, run_name, epochs=8):

    t0 = time.time()

    hist = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        verbose=1
    )

    dt = time.time() - t0

    return hist, dt


# ------------------------------------------------------------
# TODO 4: Evaluate the model
# ------------------------------------------------------------
def evaluate(model, name):

    v = model.evaluate(val_ds, verbose=0)

    t = model.evaluate(test_ds, verbose=0)

    print(f"{name} | Val acc: {v[1]:.4f} | Test acc: {t[1]:.4f}")

    return {
        "val_loss": v[0],
        "val_acc": v[1],
        "test_loss": t[0],
        "test_acc": t[1],
    }

## Q4 — Backbones & Transfer Model Builder

We will compare three pretrained ImageNet backbones using the **same head design** (GAP + Dropout + Dense):
- **VGG16**
- **ResNet50**
- **DenseNet121**

For feature extraction, keep the backbone **frozen** (`train_base=False`). For fine-tuning, unfreeze a small number of top layers with a smaller learning rate.

### Student Tasks

- Implement a function to load a pretrained backbone model (VGG16, ResNet50, or DenseNet121).

- Build a transfer learning model by adding a Global Average Pooling and classification layer.

- Optionally unfreeze the last layers of the backbone to perform fine-tuning with a smaller learning rate.


In [17]:
# ============================================================
# Question Q4 — Transfer Learning Model Construction (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build a backbone model using pretrained ImageNet weights
# 2) Support VGG16, ResNet50, and DenseNet121 architectures
# 3) Attach a classifier head using Global Average Pooling
# 4) Create a full transfer learning model
# 5) Implement fine-tuning by unfreezing part of the backbone
# ============================================================

# ------------------------------------------------------------
# TODO 1: Build backbone model
# ------------------------------------------------------------
def build_backbone(backbone_name: str):
    """Return (base_model, preprocess_fn) for a supported backbone."""

    name = backbone_name.lower()

    if name == "vgg16":
        base = tf.keras.applications.VGG16(
            weights="imagenet",
            include_top=False,
            input_shape=IMG_SIZE + (3,)
        )

        preprocess_fn = vgg16.preprocess_input

    elif name == "resnet50":
        base = tf.keras.applications.ResNet50(
            weights="imagenet",
            include_top=False,
            input_shape=IMG_SIZE + (3,)
        )

        preprocess_fn = resnet.preprocess_input

    elif name in ["densenet121", "dense121"]:
        base = tf.keras.applications.DenseNet121(
            weights="imagenet",
            include_top=False,
            input_shape=IMG_SIZE + (3,)
        )

        preprocess_fn = densenet.preprocess_input

    else:
        raise ValueError(f"Unknown backbone: {backbone_name}")

    return base, preprocess_fn


# ------------------------------------------------------------
# TODO 2: Build transfer learning model
# ------------------------------------------------------------
def build_transfer_model(backbone_name="resnet50", train_base=False, dropout=0.2):
    """Backbone + GAP head. Returns (model, base)."""

    base, preprocess_fn = build_backbone(backbone_name)

    # Freeze or unfreeze backbone
    base.trainable = bool(train_base)

    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))

    # Apply preprocessing function
    x = preprocess_fn(inputs * 255.0)

    # Forward pass through backbone
    x = base(x, training=train_base)

    # Global Average Pooling
    x = layers.GlobalAveragePooling2D()(x)

    # Dropout regularization
    x = layers.Dropout(dropout)(x)

    # Output classification layer
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    # Build final model
    model = tf.keras.Model(
        inputs,
        outputs,
        name=f"{backbone_name}_gap_{'ft' if train_base else 'fz'}"
    )

    return model, base


# ------------------------------------------------------------
# TODO 3: Fine-tune the backbone
# ------------------------------------------------------------
def fine_tune(model, base, n_unfreeze=30, lr=1e-5):
    """Unfreeze last layers of backbone and recompile model."""

    # Enable backbone training
    base.trainable = True

    if n_unfreeze is not None:
        for layer in base.layers[:-int(n_unfreeze)]:
            layer.trainable = False

    # Recompile model with smaller learning rate
    model = compile_model(model, lr=lr)

    return model

## Q5 — Model A: **VGG16** Transfer Learning (Frozen Base)
Train only a small classification head first (feature extraction).

### Student Tasks

- Build a transfer learning model using a frozen VGG16 backbone with a Global Average Pooling head.

- Compile the model and display the model summary.

- Train the model and evaluate its validation and test accuracy.

In [18]:
# ------------------------------------------------------------
# TODO 1: Build the VGG16 transfer learning model
# ------------------------------------------------------------
vgg_model, vgg_base = build_transfer_model(
    backbone_name="vgg16",
    train_base=False,
    dropout=0.2
)

# ------------------------------------------------------------
# TODO 2: Compile the model
# ------------------------------------------------------------
vgg_model = compile_model(
    vgg_model,
    lr=1e-3
)

# ------------------------------------------------------------
# TODO 3: Display model summary
# ------------------------------------------------------------
vgg_model.summary()


# ------------------------------------------------------------
# TODO 4: Train the model
# ------------------------------------------------------------
history_vgg_fz, time_vgg_fz = train_and_time(
    vgg_model,
    "vgg16_frozen",
    epochs=10
)

# ------------------------------------------------------------
# TODO 5: Evaluate the trained model
# ------------------------------------------------------------
res_vgg_fz = evaluate(
    vgg_model,
    "vgg16_frozen"
)

Model: "vgg16_gap_fz"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_7       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_3          │ (None, 128, 128,  │          0 │ input_layer_7[0]… │
│ (Multiply)          │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_6          │ (None, 128, 128)  │          0 │ multiply_3[0][0]  │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_7          │ (None, 128, 128)  │          0 │ multiply_3[0][0]  │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_8          │ (None, 128, 128)  │          0 │ multiply_3[0][0]  │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack_2 (Stack)     │ (None, 128, 128,  │          0 │ get_item_6[0][0], │
│                     │ 3)                │            │ get_item_7[0][0], │
│                     │                   │            │ get_item_8[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 128, 128,  │          0 │ stack_2[0][0]     │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vgg16 (Functional)  │ (None, 4, 4, 512) │ 14,714,688 │ add_3[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 512)       │          0 │ vgg16[0][0]       │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 512)       │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 37)        │     18,981 │ dropout_3[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 14,733,669 (56.20 MB)

 Trainable params: 18,981 (74.14 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

Epoch 1/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 14s 96ms/step - accuracy: 0.1281 - loss: 18.9694 - val_accuracy: 0.3614 - val_loss: 6.7705
Epoch 2/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 11s 92ms/step - accuracy: 0.3988 - loss: 7.5884 - val_accuracy: 0.5408 - val_loss: 4.5544
Epoch 3/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 85ms/step - accuracy: 0.5309 - loss: 5.1077 - val_accuracy: 0.6236 - val_loss: 3.6851
Epoch 4/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 11s 84ms/step - accuracy: 0.6202 - loss: 3.7759 - val_accuracy: 0.6495 - val_loss: 3.2925
Epoch 5/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 21s 98ms/step - accuracy: 0.6688 - loss: 2.8598 - val_accuracy: 0.6698 - val_loss: 2.8528
Epoch 6/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 11s 94ms/step - accuracy: 0.7069 - loss: 2.2952 - val_accuracy: 0.6861 - val_loss: 2.7377
Epoch 7/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 24s 128ms/step - accuracy: 0.7272 - loss: 1.9022 - val_accuracy: 0.6807 - val_loss: 2.6805
Epoch 8/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 11s 97ms/step - accuracy: 0.7629 - loss: 1.6657 - val_accuracy: 

## Q6 — Model B: **ResNet50** Transfer Learning (Frozen Base)
Same head design as VGG16 model for a fair backbone comparison.

### Student Tasks

- Build a transfer learning model using a frozen ResNet50 backbone with a Global Average Pooling head.

- Compile the model and display the model summary.

- Train the model and evaluate its validation and test accuracy.

In [19]:
# ------------------------------------------------------------
# TODO 1: Build the ResNet50 transfer learning model
# ------------------------------------------------------------
resnet_model, resnet_base = build_transfer_model(
    backbone_name="resnet50",
    train_base=False,
    dropout=0.2
)

# ------------------------------------------------------------
# TODO 2: Compile the model
# ------------------------------------------------------------
resnet_model = compile_model(
    resnet_model,
    lr=1e-3
)

# ------------------------------------------------------------
# TODO 3: Display model summary
# ------------------------------------------------------------
resnet_model.summary()


# ------------------------------------------------------------
# TODO 4: Train the model
# ------------------------------------------------------------
history_resnet_fz, time_resnet_fz = train_and_time(
    resnet_model,
    "resnet50_frozen",
    epochs=10
)

# ------------------------------------------------------------
# TODO 5: Evaluate the trained model
# ------------------------------------------------------------
res_resnet_fz = evaluate(
    resnet_model,
    "resnet50_frozen"
)

Model: "resnet50_gap_fz"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_9       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_4          │ (None, 128, 128,  │          0 │ input_layer_9[0]… │
│ (Multiply)          │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_9          │ (None, 128, 128)  │          0 │ multiply_4[0][0]  │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_10         │ (None, 128, 128)  │          0 │ multiply_4[0][0]  │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_11         │ (None, 128, 128)  │          0 │ multiply_4[0][0]  │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack_3 (Stack)     │ (None, 128, 128,  │          0 │ get_item_9[0][0], │
│                     │ 3)                │            │ get_item_10[0][0… │
│                     │                   │            │ get_item_11[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, 128, 128,  │          0 │ stack_3[0][0]     │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 4, 4,      │ 23,587,712 │ add_4[0][0]       │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ resnet50[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 2048)      │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 37)        │     75,813 │ dropout_4[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 23,663,525 (90.27 MB)

 Trainable params: 75,813 (296.14 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

Epoch 1/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 26s 146ms/step - accuracy: 0.4582 - loss: 2.3098 - val_accuracy: 0.7418 - val_loss: 0.8458
Epoch 2/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 88ms/step - accuracy: 0.7680 - loss: 0.7374 - val_accuracy: 0.7772 - val_loss: 0.8288
Epoch 3/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 11s 88ms/step - accuracy: 0.8421 - loss: 0.4785 - val_accuracy: 0.7636 - val_loss: 0.7928
Epoch 4/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 11s 89ms/step - accuracy: 0.8910 - loss: 0.3087 - val_accuracy: 0.7812 - val_loss: 0.7506
Epoch 5/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 89ms/step - accuracy: 0.9341 - loss: 0.2004 - val_accuracy: 0.7962 - val_loss: 0.7379
Epoch 6/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 90ms/step - accuracy: 0.9368 - loss: 0.1849 - val_accuracy: 0.7935 - val_loss: 0.8106
Epoch 7/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 89ms/step - accuracy: 0.9467 - loss: 0.1703 - val_accuracy: 0.8030 - val_loss: 0.8015
Epoch 8/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 87ms/step - accuracy: 0.9575 - loss: 0.1233 - val_accuracy: 0

## Q7 — Model C: **DenseNet121** Transfer Learning (Frozen Base)
Train a DenseNet121 backbone with the same GAP head design for a fair comparison against VGG16 and ResNet50.

### Student Tasks

- Build a transfer learning model using a frozen DenseNet121 backbone.

- Compile the model and display the model summary.

- Train the model and evaluate its validation and test accuracy.


In [20]:
# ------------------------------------------------------------
# TODO 1: Build the DenseNet121 transfer learning model
# ------------------------------------------------------------
densenet_model, densenet_base = build_transfer_model(
    backbone_name="densenet121",    # e.g., densenet121
    train_base=False,
    dropout=0.2
)

# ------------------------------------------------------------
# TODO 2: Compile the model
# ------------------------------------------------------------
densenet_model = compile_model(
    densenet_model,
    lr=1e-3
)

# ------------------------------------------------------------
# TODO 3: Display model summary
# ------------------------------------------------------------
densenet_model.summary()


# ------------------------------------------------------------
# TODO 4: Train the model
# ------------------------------------------------------------
history_densenet_fz, time_densenet_fz = train_and_time(
    densenet_model,
    "densenet121_frozen",
    epochs=10
)

# ------------------------------------------------------------
# TODO 5: Evaluate the trained model
# ------------------------------------------------------------
res_densenet_fz = evaluate(
    densenet_model,
    "densenet121_frozen"
)

Model: "densenet121_gap_fz"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_11 (InputLayer)     │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ multiply_5 (Multiply)           │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide_2 (TrueDivide)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ add_5 (Add)                     │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide_3 (TrueDivide)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ densenet121 (Functional)        │ (None, 4, 4, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_5      │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 37)             │        37,925 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,075,429 (26.99 MB)

 Trainable params: 37,925 (148.14 KB)

 Non-trainable params: 7,037,504 (26.85 MB)

Epoch 1/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 46s 223ms/step - accuracy: 0.3353 - loss: 2.9374 - val_accuracy: 0.6726 - val_loss: 1.0612
Epoch 2/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 11s 93ms/step - accuracy: 0.6885 - loss: 1.0029 - val_accuracy: 0.7636 - val_loss: 0.7961
Epoch 3/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 89ms/step - accuracy: 0.7728 - loss: 0.7001 - val_accuracy: 0.8084 - val_loss: 0.6536
Epoch 4/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 9s 78ms/step - accuracy: 0.8166 - loss: 0.5310 - val_accuracy: 0.8139 - val_loss: 0.6311
Epoch 5/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 11s 80ms/step - accuracy: 0.8550 - loss: 0.4267 - val_accuracy: 0.8356 - val_loss: 0.5637
Epoch 6/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 20s 92ms/step - accuracy: 0.8767 - loss: 0.3631 - val_accuracy: 0.8220 - val_loss: 0.6072
Epoch 7/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 11s 98ms/step - accuracy: 0.8893 - loss: 0.3086 - val_accuracy: 0.8370 - val_loss: 0.5494
Epoch 8/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 89ms/step - accuracy: 0.9144 - loss: 0.2516 - val_accuracy: 0.

## Q8 — Fine-Tuning Experiment (One Controlled Change)

Fine-tune **all three backbone models (VGG16, ResNet50, and DenseNet121)** by unfreezing only the **top N layers**.

Report whether performance **improves, stays similar, or degrades**, and briefly explain why.

**Recommended**
- Start by unfreezing the **last 10–30 layers**
- Use a **smaller learning rate** (e.g., `1e-5`)
- Use **2-3 epochs**

### Student Tasks

- Fine-tune the **VGG16, ResNet50, and DenseNet121** models by unfreezing the top layers of each backbone.

- Train each fine-tuned model for several epochs.

- Evaluate the validation and test accuracy of each fine-tuned model.

## Q8(a) — Fine-tune VGG16

In [21]:
# ------------------------------------------------------------
# TODO 1: Fine-tune the VGG16 backbone
# ------------------------------------------------------------
vgg_model = fine_tune(
    vgg_model,
    vgg_base,
    n_unfreeze=4,
    lr=1e-5
)

# ------------------------------------------------------------
# TODO 2: Train the fine-tuned model
# ------------------------------------------------------------
history_vgg_ft, time_vgg_ft = train_and_time(
    vgg_model,
    "VGG16 Fine-Tuned",
    epochs=5
)

# ------------------------------------------------------------
# TODO 3: Evaluate the fine-tuned model
# ------------------------------------------------------------
res_vgg_ft = evaluate(
    vgg_model,
    "VGG16 Fine-Tuned"
)

Epoch 1/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 16s 114ms/step - accuracy: 0.8268 - loss: 0.8594 - val_accuracy: 0.7038 - val_loss: 1.8435
Epoch 2/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 12s 106ms/step - accuracy: 0.8550 - loss: 0.5672 - val_accuracy: 0.7065 - val_loss: 1.8180
Epoch 3/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 12s 105ms/step - accuracy: 0.8821 - loss: 0.4346 - val_accuracy: 0.7160 - val_loss: 1.7796
Epoch 4/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 11s 100ms/step - accuracy: 0.9073 - loss: 0.3331 - val_accuracy: 0.7011 - val_loss: 1.6978
Epoch 5/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 11s 95ms/step - accuracy: 0.9232 - loss: 0.2532 - val_accuracy: 0.7065 - val_loss: 1.6991
VGG16 Fine-Tuned | Val acc: 0.7065 | Test acc: 0.6999


## Q8(b) — Fine-tune ResNet50

In [22]:
# ------------------------------------------------------------
# TODO 1: Fine-tune the ResNet50 backbone
# ------------------------------------------------------------
resnet_model = fine_tune(
    resnet_model,
    resnet_base,
    n_unfreeze=10,
    lr=1e-5
)

# ------------------------------------------------------------
# TODO 2: Train the fine-tuned model
# ------------------------------------------------------------
history_resnet_ft, time_resnet_ft = train_and_time(
    resnet_model,
    "ResNet50 Fine-Tuned",
    epochs=5
)

# ------------------------------------------------------------
# TODO 3: Evaluate the fine-tuned model
# ------------------------------------------------------------
res_resnet_ft = evaluate(
    resnet_model,
    "ResNet50 Fine-Tuned"
)

Epoch 1/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 27s 128ms/step - accuracy: 0.9732 - loss: 0.1490 - val_accuracy: 0.7948 - val_loss: 0.7060
Epoch 2/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 90ms/step - accuracy: 0.9810 - loss: 0.1148 - val_accuracy: 0.7935 - val_loss: 0.6949
Epoch 3/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 9s 78ms/step - accuracy: 0.9874 - loss: 0.0916 - val_accuracy: 0.7948 - val_loss: 0.6922
Epoch 4/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 11s 88ms/step - accuracy: 0.9942 - loss: 0.0766 - val_accuracy: 0.7989 - val_loss: 0.6877
Epoch 5/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 86ms/step - accuracy: 0.9922 - loss: 0.0778 - val_accuracy: 0.7989 - val_loss: 0.6830
ResNet50 Fine-Tuned | Val acc: 0.7989 | Test acc: 0.7528


## Q8(c) — Fine-tune DenseNet121

In [23]:
# ------------------------------------------------------------
# TODO 1: Fine-tune the DenseNet121 backbone
# ------------------------------------------------------------
densenet_model = fine_tune(
    densenet_model,
    densenet_base,
    n_unfreeze=8,
    lr=1e-5
)

# ------------------------------------------------------------
# TODO 2: Train the fine-tuned model
# ------------------------------------------------------------
history_densenet_ft, time_densenet_ft = train_and_time(
    densenet_model,
    "DenseNet121 Fine-Tuned",
    epochs=5
)

# ------------------------------------------------------------
# TODO 3: Evaluate the fine-tuned model
# ------------------------------------------------------------
res_densenet_ft = evaluate(
    densenet_model,
    "DenseNet121 Fine-Tuned"
)

Epoch 1/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 47s 231ms/step - accuracy: 0.9147 - loss: 0.3851 - val_accuracy: 0.8451 - val_loss: 0.5479
Epoch 2/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 86ms/step - accuracy: 0.9212 - loss: 0.3586 - val_accuracy: 0.8370 - val_loss: 0.5655
Epoch 3/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 81ms/step - accuracy: 0.9195 - loss: 0.3544 - val_accuracy: 0.8356 - val_loss: 0.5690
Epoch 4/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 11s 89ms/step - accuracy: 0.9334 - loss: 0.3327 - val_accuracy: 0.8370 - val_loss: 0.5667
Epoch 5/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 12s 90ms/step - accuracy: 0.9327 - loss: 0.3238 - val_accuracy: 0.8424 - val_loss: 0.5625
DenseNet121 Fine-Tuned | Val acc: 0.8424 | Test acc: 0.8182


## Q9 — Performance Comparison Table

#### Create a compact table comparing **frozen-backbone** models:

- VGG16 (frozen)
- ResNet50 (frozen)
- DenseNet121 (frozen)

#### Also include **fine-tuned** results for all three backbone.

### Student Tasks

- Create a comparison table summarizing the results of all models.

- Include validation accuracy, test accuracy, and training time.

- Compare frozen and fine-tuned models and identify the best-performing model.


In [24]:
# ------------------------------------------------------------
# TODO 1: Create result rows for frozen models
# ------------------------------------------------------------
rows = [

    {"Model": "VGG16 (Frozen)",
     "Val acc": res_vgg_fz["val_acc"],
     "Test acc": res_vgg_fz["test_acc"],
     "Train time (s)": time_vgg_fz},

    {"Model": "ResNet50 (Frozen)",
     "Val acc": res_resnet_fz["val_acc"],
     "Test acc": res_resnet_fz["test_acc"],
     "Train time (s)": time_resnet_fz},

    {"Model": "DenseNet121 (Frozen)",
     "Val acc": res_densenet_fz["val_acc"],
     "Test acc": res_densenet_fz["test_acc"],
     "Train time (s)": time_densenet_fz},
]

# ------------------------------------------------------------
# TODO 2: Append results for fine-tuned models if available
# ------------------------------------------------------------

if "res_vgg_ft" in globals():
    rows.append({
        "Model": "VGG16 (Fine-Tuned)",
        "Val acc": res_vgg_ft["val_acc"],
        "Test acc": res_vgg_ft["test_acc"],
        "Train time (s)": time_vgg_ft
    })

if "res_resnet_ft" in globals():
    rows.append({
        "Model": "ResNet50 (Fine-Tuned)",
        "Val acc": res_resnet_ft["val_acc"],
        "Test acc": res_resnet_ft["test_acc"],
        "Train time (s)": time_resnet_ft
    })

if "res_densenet_ft" in globals():
    rows.append({
        "Model": "DenseNet121 (Fine-Tuned)",
        "Val acc": res_densenet_ft["val_acc"],
        "Test acc": res_densenet_ft["test_acc"],
        "Train time (s)": time_densenet_ft
    })


# ------------------------------------------------------------
# TODO 3: Create comparison dataframe
# ------------------------------------------------------------
df = pd.DataFrame(rows)


# ------------------------------------------------------------
# TODO 4: Sort models by test accuracy
# ------------------------------------------------------------
df = df.sort_values("Test acc", ascending=False)

df

,Model,Val acc,Test acc,Train time (s)
5,DenseNet121 (Fine-Tuned),0.842391,0.818207,89.567634
2,DenseNet121 (Frozen),0.845109,0.816571,148.932247
1,ResNet50 (Frozen),0.800272,0.760425,121.028664
4,ResNet50 (Fine-Tuned),0.798913,0.752794,67.997644
3,VGG16 (Fine-Tuned),0.706522,0.699918,71.752336
0,VGG16 (Frozen),0.713315,0.694195,133.085372


## Short Written Questions

## **Q10** — What is the difference between *feature extraction* and *fine-tuning* in transfer learning?  
**Answer:**  Feature extraction freezes most layers and trains only the output head which makes it fast, computationally efficient and ideal for small or related datasets whereas fine-tuning unfreezes top layers and continues training for domain-specific adaptation resulting in higher accuracy but more compute.

## **Q11** — Why should the learning rate typically be lower during fine-tuning than during feature extraction?  

**Answer:** In feature extraction, most pretrained layers are frozen, so training mainly updates the new classifier head from scratch. A higher learning rate is usually okay there because we are not disturbing the pretrained feature representations.Whilst in fine-tuning, some pretrained layers are unfrozen and their weights are updated. The learning rate should usually be lower so the model makes small, careful adjustments instead of large changes that destroy useful pretrained knowledge. A high learning rate during fine-tuning can cause unstable training, overshooting, or catastrophic forgetting, especially when the new dataset is smaller than the original pretraining dataset.


## **Q12** — Give one reason why these backbones may behave differently on Oxford-IIIT Pet.  

**Answer:** it is relatively small and similar to ImageNet; pretrained features already work well, so feature extraction can perform strongly, while fine-tuning may overfit or only give small improvements.

## **Q13** — Under what conditions can fine-tuning reduce performance compared to a frozen backbone?  

**Answer:** It can reduce performance when:

* The dataset is **small**, which causes overfitting
* The **learning rate is too high**, damaging pretrained features
* The new data is **similar to the pretraining data**, so updates are unnecessary
* The model is **overtrained**, leading to loss of generalization.


### 🎉 Congratulations!

You have successfully completed **A4-TL**. Excellent work exploring and comparing **RL**, specifically **VGG16**, **VGG16**, and **DenseNet121**. Analyzing how **TL, and fine-tuning** affect performance on the **Oxford-IIIT Pet** under **CPU-only training constraints**.

### **Submission Instructions**

Please submit a **GitHub repository link** on Canvas that contains:
- The **completed Jupyter notebook**
- Notebook runs **top-to-bottom** without errors

Before submitting, ensure that:
- All **code cells (Q1–Q9)** have been executed successfully
- All **Markdown responses (Q10–Q13)** have been completed
- The notebook is **saved after execution** so that outputs are visible

Once verified, **push the final version to GitHub** and submit the repository link on Canvas.